# 02 — Download Raw Data

Download the raw GIS layers and population file into the `data/raw/` folder.

Before running this notebook, edit `scripts/config.py` and fill in:
- BOUNDARY_LAYER
- HEALTH_LAYER
- SOCIAL_LAYER
- optionally TOTAL_POP_COL if your population table includes total population

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import geopandas as gpd
import numpy as np

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from scripts.config import (
    RAW_DIR, PROCESSED_DIR, OUTPUT_DIR,
    BOUNDARIES_WFS, HEALTH_WFS, SOCIAL_WFS, POPULATION_CSV,
    METRIC_CRS, MAP_CRS, GEOGRAPHIC_CRS,
    BOUNDARY_LAYER, HEALTH_LAYER, SOCIAL_LAYER,
    MUNICIPALITY_CODE_COL, MUNICIPALITY_NAME_COL, TOTAL_POP_COL,
)
from scripts.data_sources import SOURCES
from scripts.wfs_utils import discover_wfs_layers, load_wfs_layer, download_csv, save_geodataframe, save_dataframe
from scripts.population_utils import normalize_columns, build_population_65_plus
from scripts.analysis_utils import standardize_geodataframes, build_vulnerability_index, spatial_autocorrelation, top_ranked
from scripts.plotting_utils import save_choropleth
from scripts.export_utils import export_geodataframe, export_dataframe

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
if None in (BOUNDARY_LAYER, HEALTH_LAYER, SOCIAL_LAYER):
    raise ValueError(
        "Set BOUNDARY_LAYER, HEALTH_LAYER, and SOCIAL_LAYER in scripts/config.py after Notebook 01."
    )

In [ ]:
boundaries = load_wfs_layer(BOUNDARIES_WFS, BOUNDARY_LAYER, crs=METRIC_CRS)
health = load_wfs_layer(HEALTH_WFS, HEALTH_LAYER, crs=METRIC_CRS)
social = load_wfs_layer(SOCIAL_WFS, SOCIAL_LAYER, crs=METRIC_CRS)

boundaries.to_file(RAW_DIR / "boundaries.gpkg", layer="boundaries", driver="GPKG")
health.to_file(RAW_DIR / "health.gpkg", layer="health", driver="GPKG")
social.to_file(RAW_DIR / "social.gpkg", layer="social", driver="GPKG")

print(boundaries.shape, health.shape, social.shape)

In [ ]:
pop_raw = download_csv(POPULATION_CSV, RAW_DIR / "population_56961.csv", sep=";", encoding="latin1")
print(pop_raw.head())
print(pop_raw.columns.tolist())

In [ ]:
pop_raw.to_csv(RAW_DIR / "population_raw_preview.csv", index=False)
print("Raw downloads saved in data/raw/")